## Changelog
- parent: 20260505_234035_f44a2b53
- change: replace the garage feature block with 3 PLS components fitted on log1p(SalePrice)
- hypothesis: the garage block carries strong but redundant signal (size, year,
    quality, finish, type, condition all co-move). Supervised PLS compresses the
    23-column post-encoder block into 3 components that explain ~61% of the
    variance in log-price (per eda-garage.ipynb). Replacing the block reduces
    dimensionality without losing the dominant signal and removes near-duplicate
    splits the GBM would otherwise spend depth budget on.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

    if str(comp_dir) not in sys.path:
        sys.path.insert(0, str(comp_dir))

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "eda").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")


In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)                                                                                                                                                
_outlier_mask = (                                                                                                                                                                                                                 
  train_data_raw["TotalBsmtSF"]                                                                                                                                                                                                 
  + train_data_raw["1stFlrSF"]                                                                                                                                                                                                  
  + train_data_raw["2ndFlrSF"]
) > 7000                                                                                                                                                                                                                          
train_data_raw = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

## Model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score, KFold

from utils.ames_sklearn_pipeline import (
    AmesNAImputer, AmesEncoder, TargetEncodeColumn, GaragePLSTransformer,
)
from utils.ames_feature_engineering import add_temporal_features, cast_nominal_codes, add_size_features

X = train_data_raw.drop(columns=["SalePrice"])
y = np.log1p(train_data_raw["SalePrice"])

# Pipeline built inline (not via build_pipeline) so the
# ('nominal', ...) / ('nbhd_te', ...) / ('garage_pls', ...) steps are visible.
pipe = Pipeline([
    ("na",         AmesNAImputer()),
    ("size",       FunctionTransformer(
                       add_size_features,
                       kw_args={"drop_originals": True},
                   )),
    ("fe",         FunctionTransformer(
                       add_temporal_features,
                       kw_args={"drop_originals": True},
                   )),
    ("nominal",    FunctionTransformer(cast_nominal_codes)),
    ("nbhd_te",    TargetEncodeColumn("Neighborhood")),
    ("encoder",    AmesEncoder()),
    ("garage_pls", GaragePLSTransformer(n_components=3, drop_originals=True)),  # NEW
    ("model",      GradientBoostingRegressor(
                       n_estimators=300, random_state=42,
                   )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(test_data_raw))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)